# Hugging Face + FAISS

This version uses:

- Hugging Face embeddings
- Hugging Face LLM
- FAISS vector database
- Contextual Compression Retriever

In [ ]:
# pip install -U langchain langchain-community langchain-huggingface faiss-cpu transformers torch sentence-transformers accelerate

import os

from langchain_huggingface import (
    HuggingFaceEndpoint,
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import FAISS

from langchain_core.documents import Document

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor


# --------------------------------------------------
# 1. Hugging Face API KEY
# --------------------------------------------------

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"


# --------------------------------------------------
# 2. Hugging Face LLM
# --------------------------------------------------

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct",
    task="text-generation",
    max_new_tokens=256,
    temperature=0.2
)


# --------------------------------------------------
# 3. Hugging Face Embedding Model
# --------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# --------------------------------------------------
# 4. Documents
# --------------------------------------------------

documents = [

    Document(
        page_content="""
        Transformers are neural network architectures widely used
        in Natural Language Processing. They use attention mechanisms
        to understand relationships between tokens.
        """
    ),

    Document(
        page_content="""
        Self-attention allows every token in a sequence to interact
        with other tokens. It uses Query, Key and Value vectors to
        calculate attention scores.
        """
    ),

    Document(
        page_content="""
        Positional encoding provides information about the position
        of tokens because Transformers do not process tokens
        sequentially like traditional RNNs.
        """
    ),

    Document(
        page_content="""
        RNNs process sequences sequentially and maintain a hidden
        state that carries information from previous time steps.
        """
    ),

]


# --------------------------------------------------
# 5. Create FAISS Vector Store
# --------------------------------------------------

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)


# --------------------------------------------------
# 6. Base Retriever
# --------------------------------------------------

base_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3
    }
)


# --------------------------------------------------
# 7. Create Compressor
# --------------------------------------------------

compressor = LLMChainExtractor.from_llm(
    llm
)


# --------------------------------------------------
# 8. Contextual Compression Retriever
# --------------------------------------------------

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)


# --------------------------------------------------
# 9. Query
# --------------------------------------------------

query = "What is self-attention in Transformers?"


# --------------------------------------------------
# 10. Retrieve Compressed Documents
# --------------------------------------------------

docs = compression_retriever.invoke(query)


# --------------------------------------------------
# 11. Display Results
# --------------------------------------------------

print("\nCompressed Documents:\n")

for i, doc in enumerate(docs, start=1):

    print(f"--- Document {i} ---")

    print(doc.page_content)

    print()